# 参考記事

[3次元有限要素法をPythonで実装する(四面体要素)](https://qiita.com/Altaka4128/items/41101c96729b68d7c96f)

本計算は以下の点を考慮している

- 任意の形状で構造解析ができる（CATIAからinpファイルをエクスポート）
- 荷重条件、固定条件もinpファイルでエクスポート
- displacementをvtkファイルに書き出してParaViewでの可視化ができる
- 大きな行列を取り扱うため疎行列（0の行列要素が多い行列）はscipyのsparseで行列を圧縮している

# 全体のプログラム

In [1]:
import numpy as np
import numpy.linalg as LA
import pandas as pd
import pathlib
#import Data_import_lib.Data_import as di
import Data_import_lib

#from pyevtk.hl import pointsToVTK
import gc
#import dask
#import dask.array as da

import time

from scipy import sparse
from scipy.sparse.linalg import inv


#FC300
#THICKNESS = 0.1                                             #要素の厚さ
YOUNG = 130000.0                                            #ヤング率(MPa)
POISSON = 0.27                                               #ポアソン比
LAMBDA = 11.9                                               #線膨張係数（/K）

NODE_TRIA3 = 4                                              #要素の接点数
COMPONENTS = 6                                              #ひずみと応力の成分数
DOF_NODE = 3                                                #接点自由度
weight = 1.0 / 6.0                                          #積分点の重み係数


#荷重条件
Fx = -100.0                                                    #荷重（X方向）
Fy = 0.0                                                    #荷重（Y方向）
Fz = 0.0                                                    #荷重（Z方向）
BX = 0.0                                                    #物体力（X方向）
BY = 0.0                                                    #物体力（Y方向）
BZ = 0.0                                                    #物体力（Z方向）
face_px = 0.0                                               #表面力（X方向）
face_py = 1.0                                               #表面力（Y方向）
face_pz = 0.0                                               #表面力（Ｚ方向）

inpfileName = "Quad4_FEM_00.inp"                            #inpファイル名
#inpfileName = "Quad4_FEM_B15_BED.inp" 

# 初期状態
def initialize():
    #FEMモデル情報取得（Node,Element）
    fem_info= Data_import_lib.Data_import.data_import(inpfileName)
    
    #[0]:node, [1]:element, [2]:fixed, [3]:face_load [4]:face_load_element [5]:foece
    #モジュール定数
    NODES=len(fem_info[0])                                          #全節点数
    ELEMENTS=len(fem_info[1])                                       #全要素数
    DOF_TOTAL = DOF_NODE * NODES                                    #モデル全体の自由度
    DOF_TRIA3 = NODE_TRIA3 * DOF_NODE                               #要素自由度
    
    #モジュールレベル変数
    #x=np.zeros((NODES),dtype="float32")                                               #接点のＸ座標配列
    x=sparse.lil_matrix((1, NODES), dtype="float32")
    #y=np.zeros((NODES),dtype="float32")                                               #接点のＹ座標配列
    y=sparse.lil_matrix((1, NODES), dtype="float32")
    #z=np.zeros((NODES),dtype="float32")                                               #節点のＺ座標配列
    z=sparse.lil_matrix((1,NODES), dtype="float32")
    #Input Data
    for i in range(NODES):
        x[0,i]=fem_info[0][i][1]
        y[0,i]=fem_info[0][i][2]
        z[0,i]=fem_info[0][i][3]
        #print("NODE:",i,"x:",x[0,i],"y:",y[0,i]) #--- ok
        
    #要素内節点順配列
    #connectivity=np.zeros((ELEMENTS,NODE_TRIA3),dtype="int32")
    connectivity=sparse.lil_matrix((ELEMENTS, NODE_TRIA3),dtype="int32")      
    #Input Data
    for e in range(ELEMENTS):
        for i in range(NODE_TRIA3):
            connectivity[e,i]=fem_info[1][e][i+1]
        #print("ELEMENT:",e,"POINT(",connectivity[e],")") #--- ok
    
    #拘束点（X＝Y＝0とする）
    #fixed=np.zeros(len(fem_info[2]),dtype="int32")
    fixed=sparse.lil_matrix((1, len(fem_info[2])),dtype="int32")
    for i in range(len(fem_info[2])):
        fixed[0,i]=fem_info[2][i]
        #print("***fixed***:",fixed[0,i])

    #U=np.zeros(DOF_TOTAL,dtype="int32")
    U = sparse.lil_matrix((1, DOF_TOTAL), dtype="int32")
    Um=np.zeros(DOF_TOTAL,dtype="bool")
    for i in range(len(fem_info[2])):
        #for j in range(len(fem_info[2][1])):
        for k in range(DOF_NODE):
            Um[fixed[0,i]*DOF_NODE+k]=True                   #拘束されているｘ、ｙ、ｚをＴｒｕｅとする（全体行列系に）
            #print(f"***fixed_NODE:{fixed[0,i]*DOF_NODE+k}***:",fixed[0,i])
    
    #外力_1(１節点のみ)
    #force=np.zeros(len(fem_info[5]),dtype="int32")
    force=sparse.lil_matrix((1, len(fem_info[3])),dtype="int32")
    for i in range(len(fem_info[3])):
        force[0,i]=fem_info[3][i]
        print("***force***:",force[0,i])
    
    #表面力
    #face_load=np.zeros(len(fem_info[3]),dtype="int32")
    face_load=sparse.lil_matrix((1, len(fem_info[4])),dtype="int32")
    for i in range(len(fem_info[4])):
        face_load[0,i]=fem_info[4][i]
        #print("***face_load***:",face_load[0,i])

    

    #Face_ELEMENT
    #face_load_element=np.zeros(len(fem_info[4]),dtype="int32")
    face_load_element=sparse.lil_matrix((1, len(fem_info[5])),dtype="int32")
    for i in range(len(fem_info[5])):
        face_load[0,i]=fem_info[5][i]
        #print("***face_load_element***:",face_load_element[0,i])

    return x, y, z, connectivity, NODES, ELEMENTS, DOF_TOTAL, DOF_TRIA3, U, Um, force, face_load, face_load_element
    
    
# Dマトリクス
def make_D():
    #D=np.zeros((COMPONENTS, COMPONENTS),dtype="float32")
    #coef = YOUNG / (1 - 2*POISSON) / (1 + POISSON)
    #D=np.array([
    #            [coef*(1-POISSON), coef*POISSON, coef*POISSON, 0, 0, 0],
    #            [coef*POISSON, coef*(1-POISSON), coef*POISSON, 0, 0, 0],
    #            [coef*POISSON, coef*POISSON, coef*(1-POISSON), 0, 0, 0],
    #            [0, 0, 0, coef*(1-2*POISSON)/2, 0, 0],
    #            [0, 0, 0, 0, coef*(1-2*POISSON)/2, 0],
    #            [0, 0, 0, 0, 0, coef*(1-2*POISSON)/2]
    #])

    coef = YOUNG / (1.0 - 2.0*POISSON) / (1.0 + POISSON)
    D_matrix=([
        [coef*(1.0-POISSON), coef*POISSON, coef*POISSON, 0.0, 0.0, 0.0],
        [coef*POISSON, coef*(1.0-POISSON), coef*POISSON, 0.0, 0.0, 0.0],
        [coef*POISSON, coef*POISSON, coef*(1.0-POISSON), 0.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, coef*(1.0-2.0*POISSON)/2.0, 0.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, coef*(1.0-2.0*POISSON)/2.0, 0.0],
        [0.0, 0.0, 0.0, 0.0, 0.0, coef*(1.0-2.0*POISSON)/2.0]
        ])
    
    D = sparse.lil_matrix(D_matrix)
    return D

# Bマトリクス
def make_B():
    B = np.zeros((ELEMENTS, COMPONENTS, DOF_TRIA3),dtype="float32")
    Jmat = np.zeros((ELEMENTS, DOF_NODE, DOF_NODE),dtype="float32")  

    for e in range(ELEMENTS):
        x0, y0, z0 = x[0, connectivity[e,0]], y[0, connectivity[e,0]], z[0, connectivity[e,0]]
        x1, y1, z1 = x[0, connectivity[e,1]], y[0, connectivity[e,1]], z[0, connectivity[e,1]]
        x2, y2, z2 = x[0, connectivity[e,2]], y[0, connectivity[e,2]], z[0, connectivity[e,2]]
        x3, y3, z3 = x[0, connectivity[e,3]], y[0, connectivity[e,3]], z[0, connectivity[e,3]]
        #print(f"x0:{x0}, y0:{y0}, z0:{z0}")

        #➀形状関数の正規化座標による偏微分
        # N1=1-a-b-c, N2=a, N3=b, N4=c 
        dN1da = -1.0; dN2da = 1.0; dN3da = 0.0; dN4da = 0.0
        dN1db = -1.0; dN2db = 0.0; dN3db = 1.0; dN4db = 0.0
        dN1dc = -1.0; dN2dc = 0.0; dN3dc = 0.0; dN4dc = 1.0

        #➁座標成分を正規化座標成分で偏微分
        dxda = dN1da*x0 + dN2da*x1 + dN3da*x2 + dN4da*x3
        dyda = dN1da*y0 + dN2da*y1 + dN3da*y2 + dN4da*y3
        dzda = dN1da*z0 + dN2da*z1 + dN3da*z2 + dN4da*z3
        dxdb = dN1db*x0 + dN2db*x1 + dN3db*x2 + dN4db*x3
        dydb = dN1db*y0 + dN2db*y1 + dN3db*y2 + dN4db*y3
        dzdb = dN1db*z0 + dN2db*z1 + dN3db*z2 + dN4db*z3
        dxdc = dN1dc*x0 + dN2dc*x1 + dN3dc*x2 + dN4dc*x3
        dydc = dN1dc*y0 + dN2dc*y1 + dN3dc*y2 + dN4dc*y3
        dzdc = dN1dc*z0 + dN2dc*z1 + dN3dc*z2 + dN4dc*z3

        # ➂ヤコビ行列Jmat
        Jmat[e]=np.array([
                        [dxda, dyda, dzda],
                        [dxdb, dydb, dzdb],
                        [dxdc, dydc, dzdc]
                        ])
        
        # ∂Ｎｉ／∂ａ、∂Ｎｉ／∂ｂ、∂Ｎｉ／∂ｃ
        dN1dabc = [dN1da, dN1db, dN1dc]; dN2dabc = [dN2da, dN2db, dN2dc]; dN3dabc = [dN3da, dN3db, dN3dc]; dN4dabc = [dN4da, dN4db, dN4dc]
        
        # ∂Ｎｉ／∂ｘ、∂Ｎｉ／∂ｙ、∂Ｎｉ／∂ｚ
        dN1dxyz = LA.solve(Jmat[e], dN1dabc); dN2dxyz = LA.solve(Jmat[e], dN2dabc); dN3dxyz = LA.solve(Jmat[e], dN3dabc); dN4dxyz = LA.solve(Jmat[e], dN4dabc)

        # 要素e番目の行列
        dN1dx = dN1dxyz[0]; dN2dx = dN2dxyz[0]; dN3dx = dN3dxyz[0]; dN4dx = dN4dxyz[0]
        dN1dy = dN1dxyz[1]; dN2dy = dN2dxyz[1]; dN3dy = dN3dxyz[1]; dN4dy = dN4dxyz[1]
        dN1dz = dN1dxyz[2]; dN2dz = dN2dxyz[2]; dN3dz = dN3dxyz[2]; dN4dz = dN4dxyz[2]
        
        B[e] =  np.array([
                        [dN1dx, 0.0, 0.0, dN2dx, 0.0, 0.0, dN3dx, 0.0, 0.0, dN4dx, 0.0, 0.0],
                        [0.0, dN1dy, 0.0, 0.0, dN2dy, 0.0, 0.0, dN3dy, 0.0, 0.0, dN4dy, 0.0],
                        [0.0, 0.0, dN1dz, 0.0, 0.0, dN2dz, 0.0, 0.0, dN3dz, 0.0, 0.0, dN4dz],
                        [0.0, dN1dz, dN1dy, 0.0, dN2dz, dN2dy, 0.0, dN3dz, dN3dy, 0.0, dN4dz, dN4dy],
                        [dN1dz, 0.0, dN1dx, dN2dz, 0.0, dN2dx, dN3dz, 0.0, dN3dx, dN4dz, 0.0, dN4dx],
                        [dN1dy, dN1dx, 0.0, dN2dy, dN2dx, 0.0, dN3dy, dN3dx, 0.0, dN4dy, dN4dx, 0.0]
                        ])
        # print("Pass:", e)
        
    """
    print("==============  Bマトリクス ========================")
    for e in range(ELEMENTS):
        print(f"======= 要素{e}番目==========")
        print(f"         B[{e}] = {B[e]}")
    """
    return B, Jmat
    #return B, volume_elements
    
#  要素剛性マトリクス
def make_Ke():
    Ke= np.zeros((ELEMENTS,  DOF_TRIA3, DOF_TRIA3),dtype="float32") 
    for e in range(ELEMENTS):
        Ke[e] = weight*B[e].T @ D @ B[e] * LA.det(Jmat[e])
        #print(Ke[e])
    
    return Ke

# 全体剛性マトリクス
def make_K():
    #K= np.zeros((DOF_TOTAL, DOF_TOTAL),dtype="float32")
    K= sparse.lil_matrix((DOF_TOTAL, DOF_TOTAL),dtype="float32")

    for e in range(ELEMENTS):
        for r in range(DOF_TRIA3):
            rt = ((connectivity[e, r // DOF_NODE]+1)*DOF_NODE-((r+1)%DOF_NODE))-1
            #print("rt:",rt)
            for c in range(DOF_TRIA3):
                ct = ((connectivity[e, c // DOF_NODE]+1)*DOF_NODE-((c+1)%DOF_NODE))-1
                K[rt,ct] = K[rt,ct] + Ke[e,r,c]

    return K
    

#荷重
def calc_force_tria3():
    #F=np.zeros(DOF_TOTAL,dtype="float32")
    F=sparse.lil_matrix((1, DOF_TOTAL),dtype="float32")

    #for i in range(DOF_TOTAL):
    #    F[0, i] = 0.0

    #note: forceは１点としている
    F[0, force[0,0]*DOF_NODE] += Fx    
    F[0, force[0,0]*DOF_NODE+1] += Fy
    F[0, force[0,0]*DOF_NODE+2] += Fz

    return F


#物体力 *** 未完成　***
def calc_body_force_tria3(U):
    for e in range(ELEMENTS):        
        for m in range(NODE_TRIA3):
            n = connectivity[e, m]
            F[n * 2] = F[n * 2] + THICKNESS* area_elements[e]/ 3* BX
            F[(n * 2)+1] = F[(n * 2)+1] + THICKNESS* area_elements[e]/ 3* BY

    return F


#表面力　***　未完成　***
def calc_surface_force_tria3():
    #表面力が作用している辺を調べる・・・節点が含まれない条件で辺を検知
    #print(face_load_element)
    for i, e in enumerate(face_load_element):
        if not connectivity[e,0] in face_load:
            na = connectivity[e, 1]
            nb = connectivity[e, 2]
            print("e:",e,"edge2")
        if not connectivity[e,1] in face_load:
            na = connectivity[e, 2]
            nb = connectivity[e, 0]
            print("e:",e,"edge3")
        if not connectivity[e,2] in face_load:
            na = connectivity[e, 0]
            nb = connectivity[e, 1]
            print("e:",e,"edge1")
        
        xa = x[na]; ya = y[na]
        xb = x[nb]; yb = y[nb]
        edge_length = np.sqrt((xa - xb) * (xa - xb) + (ya - yb) * (ya - yb))
        fx = face_px * edge_length / 2
        fy = face_py * edge_length / 2
        
        F[na * 2] = F[na * 2] + fx
        F[(na * 2)+1] = F[(na * 2)+1] + fy
        F[nb * 2] = F[nb * 2] + fx
        F[(nb * 2)+1] = F[(nb * 2)+1] + fy
        
    return F

#境界条件処理
def set_baoudary_U_F():
    
    # 全体行列をコピー
    #Kc = sparse.lil_matrix.copy(K)

    for r in range(DOF_TOTAL):
        if Um[r] == True:
            for rr in range(DOF_TOTAL): # 変位拘束が存在する行の成分を０にする
                if rr != r:
                    F[0, rr] -= K[rr,r]*U[0, r]                    
            for rr in range(DOF_TOTAL):
                K[rr,r] = 0.0
            for cc in range(DOF_TOTAL): # 変位拘束が存在する列の成分を０にする
                K[r,cc] = 0.0

            K[r,r] = 1.0 #対角成分を１

            #F[r] = U[r]
            F[0, r] = U[0, r]

    return U, F ,K

#逆行列を求める（ＣＳＣ形式）
def to_csc():
    K_csc = K.tocsc()
    F_csc = F.tocsc()
    Inv_K_csc = inv(K_csc)    
    FT_csc = F_csc.T

    del K_csc
    del F_csc
    gc.collect

    return Inv_K_csc, FT_csc

#逆行列で計算
def solve():

    #K_csc = K.tocsc()
    #F_csc = F.tocsc()

    #Ua=inv(K_csc)@F_csc.T
    Ua=Inv_K_csc@FT_csc

    #Ua = LA.inv(Kc) @ F
    
    
    return Ua

#Paraviewフォーマットに値を出力
def outputvtk():
    file_name = f"{inpfileName[:-4]}_TRIA_3.vtk"
    #f_path = pathlib.Path(__file__).parent.resolve() / file_name 
    import os
    PWD = os.getcwd()
    f_path = f"{PWD}/{file_name}"

    with open(f_path, mode = "w") as f:
        
        #Header出力
        print("# vtk DataFile Version 2.0",file=f)
        print("Header",file=f)
        print("ASCII",file=f)
        print("DATASET UNSTRUCTURED_GRID",file=f)
        print(" ",file=f)
        
        #節点座標出力
        print("POINTS", NODES, " double",file=f)
        for i in range(NODES):
            print(x[0,i]," ",y[0,i]," ",z[0,i],file=f)
        print(" ",file=f)
        
        #要素構成節点番号出力
        print("CELLS", ELEMENTS, ELEMENTS*5,file=f)
        for i in range(ELEMENTS):
            print(4," ",end="",file=f)
            for j in range(NODE_TRIA3):
                print(connectivity[i,j]," ",end="",file=f)
            print("",file=f)
        print(" ",file=f)
        
        #要素タイプ出力
        print("CELL_TYPES", ELEMENTS,file=f)
        for i in range(ELEMENTS):
            print(10,file=f)
        print(" ",file=f)
        
        #節点応力出力
        print("POINT_DATA", NODES,file=f)
        #print("SCALARS Sx float 1",file=f)
        #print("LOOKUP_TABLE default",file=f)
        #for i in range(NODES):
        #    print(stress_node[i,0],file=f)
        print(" ",file=f)
        
        #節点変位出力
        print("VECTORS Displacement float",file=f)
        for i in range(NODES):
            print(Ua[i*DOF_NODE,0]," ",Ua[i*DOF_NODE+1,0]," ",Ua[i*DOF_NODE+2,0],file=f)
        print(" ",file=f)




if __name__ == "__main__":
    #計測開始
    t1 = time.time()
    #初期化
    x, y, z, connectivity, NODES, ELEMENTS, DOF_TOTAL, DOF_TRIA3, U, Um, force, face_load, face_load_element=  initialize()
    #print("conectivity: ", connectivity[0])

    #Dマトリクス --- ok
    D = make_D()
    print("*** PASS_make_D ***")

    #Bマトリクス --- ok
    B, Jmat = make_B()
    print("*** PASS_make_B ***")

    #要素剛性行列 --- ok
    Ke = make_Ke()
    print("*** make_Ke ***")


    #全体剛性行列 --- ok
    K = make_K()
    print("*** PASS_make_K ***")

    del Ke
    del B
    del Jmat
    gc.collect

    #荷重 --- check_ok
    F = calc_force_tria3()
    print("*** PASS_calc_force_tria3 ***")

#    #物体力
#    F = calc_body_force_tria3()
#    
#    #表面力 --- check_ok
#    F=calc_surface_force_tria3()
#    #print("F:",F)

    #境界条件処理
    #U, F, Kc = set_baoudary_U_F()
    U, F, K = set_baoudary_U_F()
    print("*** PASS_set_boundary_U_F ***")

    #メモリー消費対策
    Inv_K_csc, FT_csc = to_csc()
    del K
    del F
    gc.collect
    print("*** to_csc ***")

    #ソルバ―➀：逆行列で計算
    Ua = solve()
    print("*** PASS_solve ***")
    
    #Vtk出力
    outputvtk()

    #計測終了
    t2 = time.time()

    elapsed_time = t2 - t1
    print(f"経過時間：{elapsed_time}")

====実行された====
***force***: 282
*** PASS_make_D ***
*** PASS_make_B ***
*** make_Ke ***
*** PASS_make_K ***
*** PASS_calc_force_tria3 ***
*** PASS_set_boundary_U_F ***
*** to_csc ***
*** PASS_solve ***
経過時間：3.180948257446289


## ParaViewでvtkファイルを可視化
<img src="img/3次元1次要素梁の問題.png">

In [112]:
print(Ua.toarray())

[[ 0.    ]
 [ 0.    ]
 [ 0.    ]
 [-0.0109]
 [-0.0001]
 [-0.0016]
 [ 0.    ]
 [ 0.    ]
 [ 0.    ]
 [-0.0109]
 [-0.0001]
 [-0.0016]
 [ 0.    ]
 [ 0.    ]
 [ 0.    ]
 [-0.0109]
 [-0.0001]
 [ 0.0016]
 [ 0.    ]
 [ 0.    ]
 [ 0.    ]
 [-0.0109]
 [-0.0001]
 [ 0.0016]
 [-0.0001]
 [-0.0001]
 [-0.0002]
 [-0.0002]
 [-0.0001]
 [-0.0003]
 [-0.0005]
 [-0.0001]
 [-0.0005]
 [-0.0009]
 [-0.0001]
 [-0.0007]
 [-0.0014]
 [-0.0001]
 [-0.0008]
 [-0.0021]
 [-0.0001]
 [-0.001 ]
 [-0.0027]
 [-0.0001]
 [-0.0011]
 [-0.0037]
 [-0.0001]
 [-0.0012]
 [-0.0043]
 [-0.0001]
 [-0.0013]
 [-0.0054]
 [-0.0001]
 [-0.0014]
 [-0.0062]
 [-0.0001]
 [-0.0015]
 [-0.0075]
 [-0.0001]
 [-0.0015]
 [-0.0083]
 [-0.0001]
 [-0.0016]
 [-0.0093]
 [-0.0001]
 [-0.0016]
 [-0.0101]
 [-0.0001]
 [-0.0016]
 [ 0.    ]
 [ 0.    ]
 [ 0.    ]
 [ 0.    ]
 [ 0.    ]
 [ 0.    ]
 [ 0.    ]
 [ 0.    ]
 [ 0.    ]
 [-0.0001]
 [ 0.0001]
 [-0.0002]
 [-0.0002]
 [ 0.0001]
 [-0.0003]
 [-0.0005]
 [ 0.0001]
 [-0.0005]
 [-0.0009]
 [ 0.    ]
 [-0.0006]
 [-0.0014]

# プログラムの中身を理解する

## 有限要素法の理解

## ライブラリのインポート

In [2]:
import numpy as np
import numpy.linalg as LA
import pandas as pd
import pathlib
#import Data_import_lib.Data_import as di
import Data_import_lib

#from pyevtk.hl import pointsToVTK
import gc
#import dask
#import dask.array as da

import time

from scipy import sparse
from scipy.sparse.linalg import inv

# 変数を定義

In [3]:
#FC300
#THICKNESS = 0.1                                             #要素の厚さ
YOUNG = 130000.0                                            #ヤング率(MPa)
POISSON = 0.27                                               #ポアソン比
LAMBDA = 11.9                                               #線膨張係数（/K）

NODE_TRIA3 = 4                                              #要素の接点数
COMPONENTS = 6                                              #ひずみと応力の成分数
DOF_NODE = 3                                                #接点自由度
weight = 1.0 / 6.0                                          #積分点の重み係数


#荷重条件
Fx = -100.0                                                    #荷重（X方向）
Fy = 0.0                                                    #荷重（Y方向）
Fz = 0.0                                                    #荷重（Z方向）
BX = 0.0                                                    #物体力（X方向）
BY = 0.0                                                    #物体力（Y方向）
BZ = 0.0                                                    #物体力（Z方向）
face_px = 0.0                                               #表面力（X方向）
face_py = 1.0                                               #表面力（Y方向）
face_pz = 0.0                                               #表面力（Ｚ方向）

inpfileName = "Quad4_FEM_00.inp"                            #inpファイル名
#inpfileName = "Quad4_FEM_B15_BED.inp" 

# 初期状態の定義

ライブラリを変更した

Data_input/Data_import.py

In [4]:
# class Data_import:

#     @classmethod    
#     def data_import(cls, inpfileName):
#         file_name_0 = "settings.yml"
#         file_name_1 = "./Inp_Data/" + inpfileName#"Quad4_FEM.inp"

節点番号と節点座標を取得する

In [5]:
# 初期状態
#FEMモデル情報取得（Node,Element）
fem_info= Data_import_lib.Data_import.data_import(inpfileName)

In [6]:
# fem_info[0]が節点に対応
print(len(fem_info[0]))
fem_info[0][:10]

425


[[0, ' 0.', ' 0.', ' 0.\n'],
 [1, ' 0.', ' 0.', ' 150.\n'],
 [2, ' 30.', ' 0.', ' 0.\n'],
 [3, ' 30.', ' 0.', ' 150.\n'],
 [4, ' 30.', ' 30.', ' 0.\n'],
 [5, ' 30.', ' 30.', ' 150.\n'],
 [6, ' 0.', ' 30.', ' 0.\n'],
 [7, ' 0.', ' 30.', ' 150.\n'],
 [8, ' 0.', ' 0.', ' 8.62450337052276\n'],
 [9, ' 0.', ' 0.', ' 15.3315896739104\n']]

In [7]:
# fem_info[1]が要素に対応
fem_info[1][:10]

[[0, 0, 23, 88, 8],
 [1, 92, 23, 8, 290],
 [2, 280, 88, 23, 290],
 [3, 290, 88, 23, 8],
 [4, 8, 290, 88, 271],
 [5, 277, 280, 24, 290],
 [6, 290, 95, 24, 291],
 [7, 290, 23, 280, 24],
 [8, 24, 290, 23, 92],
 [9, 290, 277, 280, 279]]

In [8]:
# fem_info[2]は拘束点
fem_info[2][:10]

[0, 2, 4, 6, 23, 24, 25, 44, 45, 46]

In [9]:
# 外力_1(１節点のみ)
fem_info[3]

[282]

In [10]:
# fem_info[4]は表面力
fem_info[4]

[0]

# Fortranで使用するためにテキストで出力する

```python
fem_info[i][j][k]
```

- i = 0:座標, 1:要素, 2:荷重節点, 3:節点外力, 4:表面力
- j = 0:番号
- k = 0:要素番号

In [11]:
xpos_filename = "xpos.txt"
ypos_filename = "ypos.txt"
zpos_filename = "zpos.txt"

def writepos(pos_filename, i):
    with open(pos_filename, mode='w') as f:
        for n in fem_info[0]:
            if pos_filename == "zpos_filename":
                f.write(n[i])
            else:
                f.write(f"{n[i]}\n")

    f.close()

writepos(xpos_filename,1) #x座標の書き出し
writepos(ypos_filename,2) #x座標の書き出し
writepos(zpos_filename,3) #x座標の書き出し

In [12]:
fem_info[1][:5]

[[0, 0, 23, 88, 8],
 [1, 92, 23, 8, 290],
 [2, 280, 88, 23, 290],
 [3, 290, 88, 23, 8],
 [4, 8, 290, 88, 271]]

In [13]:
for n in fem_info[1][:5]:
    print(str(n)[1:-1])

0, 0, 23, 88, 8
1, 92, 23, 8, 290
2, 280, 88, 23, 290
3, 290, 88, 23, 8
4, 8, 290, 88, 271


In [14]:
len(fem_info[1])

1403

In [15]:
element_filename = "element.txt"

def writepos(pos_filename, i):
    with open(pos_filename, mode='w') as f:
        for n in fem_info[1]:
            f.write(f"{str(n)[1:-1]}\n")
    f.close()

writepos(element_filename,1) #x座標の書き出し

出力された

- xpos.txt
- ypos.txt
- zpos.txt
- element.txt

Fortranで計算する際に使用する

節点の自由度からx,y,zの配列を決定する

次に拘束点のFortran用出力

In [16]:
fem_info[2]

[0,
 2,
 4,
 6,
 23,
 24,
 25,
 44,
 45,
 46,
 65,
 66,
 67,
 86,
 87,
 88,
 272,
 273,
 274,
 275,
 276,
 277,
 278,
 279,
 280]

In [17]:
fixed_filename = "fixed_filename.txt"

with open(fixed_filename, mode='w') as f:
    for n in fem_info[2]:
        f.write(f"{str(n)}\n")

f.close()

In [18]:
#[0]:node, [1]:element, [2]:fixed, [3]:face_load [4]:face_load_element [5]:foece
#モジュール定数
NODES=len(fem_info[0])                                          #全節点数
ELEMENTS=len(fem_info[1])                                       #全要素数
DOF_TOTAL = DOF_NODE * NODES                                    #モデル全体の自由度
DOF_TRIA3 = NODE_TRIA3 * DOF_NODE                               #要素自由度

#モジュールレベル変数
#x=np.zeros((NODES),dtype="float32")                                               #接点のＸ座標配列
x=sparse.lil_matrix((1, NODES), dtype="float32")
#y=np.zeros((NODES),dtype="float32")                                               #接点のＹ座標配列
y=sparse.lil_matrix((1, NODES), dtype="float32")
#z=np.zeros((NODES),dtype="float32")                                               #節点のＺ座標配列
z=sparse.lil_matrix((1,NODES), dtype="float32")

In [19]:
print(x)

有限要素法の計算は行列内に0を多く含むため、疎行列（スパース行列）を効率的に取り扱うことが重要。

[Python, SciPy（scipy.sparse）で疎行列を生成・変換](https://note.nkmk.me/python-scipy-sparse-matrix-csr-csc-coo-lil/)

[Sparse matrices (scipy.sparse) — SciPy v1.3.0 Reference Guide
](https://docs.scipy.org/doc/scipy/reference/sparse.html)


成分のほとんどが0である疎行列はリストやnumpy.ndarrayで表すよりも、scipy.sparseのクラスで表したほうがメモリ使用量が少なく処理速度も高速になる。

x, y, z座標の値を取得する

In [20]:
#Input Data
for i in range(NODES):
    x[0,i]=fem_info[0][i][1]
    y[0,i]=fem_info[0][i][2]
    z[0,i]=fem_info[0][i][3]
    #print("NODE:",i,"x:",x[0,i],"y:",y[0,i]) #--- ok

要素に対して順番を付ける

<img src="img/3次元1次要素の要素と節点.PNG">

In [21]:
NODE_TRIA3

4

In [22]:
#要素内節点順配列
#connectivity=np.zeros((ELEMENTS,NODE_TRIA3),dtype="int32")
connectivity=sparse.lil_matrix((ELEMENTS, NODE_TRIA3),dtype="int32")      
#Input Data
for e in range(ELEMENTS):
    for i in range(NODE_TRIA3):
        connectivity[e,i]=fem_info[1][e][i+1]
    #print("ELEMENT:",e,"POINT(",connectivity[e],")") #--- ok

In [23]:
connectivity

<1403x4 sparse matrix of type '<class 'numpy.int32'>'
	with 5611 stored elements in List of Lists format>

<img src="img/3次元1次要素の荷重と固定.PNG">

拘束条件を加える

In [24]:
#拘束点（X＝Y＝0とする）
#fixed=np.zeros(len(fem_info[2]),dtype="int32")
fixed=sparse.lil_matrix((1, len(fem_info[2])),dtype="int32")
for i in range(len(fem_info[2])):
    fixed[0,i]=fem_info[2][i]
    #print("***fixed***:",fixed[0,i])

#U=np.zeros(DOF_TOTAL,dtype="int32")
U = sparse.lil_matrix((1, DOF_TOTAL), dtype="int32")

In [25]:
fixed

<1x25 sparse matrix of type '<class 'numpy.int32'>'
	with 24 stored elements in List of Lists format>

In [26]:
Um=np.zeros(DOF_TOTAL,dtype="bool")
for i in range(len(fem_info[2])):
    #for j in range(len(fem_info[2][1])):
    for k in range(DOF_NODE):
        Um[fixed[0,i]*DOF_NODE+k]=True                   #拘束されているｘ、ｙ、ｚをＴｒｕｅとする（全体行列系に）
        #print(f"***fixed_NODE:{fixed[0,i]*DOF_NODE+k}***:",fixed[0,i])

In [27]:
print(Um)

[ True  True  True ... False False False]


Fortran用出力

In [28]:
Um_filename = "Um_filename.txt"

with open(Um_filename, mode='w') as f:
    for n in Um:
        if n == False:
            f.write("0\n")
        else:
            f.write("1\n")

f.close()

In [29]:
#外力_1(１節点のみ)
#force=np.zeros(len(fem_info[5]),dtype="int32")
force=sparse.lil_matrix((1, len(fem_info[3])),dtype="int32")
for i in range(len(fem_info[3])):
    force[0,i]=fem_info[3][i]
    print("***force***:",force[0,i])

***force***: 282


In [30]:
print(force)

  (0, 0)	282


In [31]:
#表面力
#face_load=np.zeros(len(fem_info[3]),dtype="int32")
face_load=sparse.lil_matrix((1, len(fem_info[4])),dtype="int32")
for i in range(len(fem_info[4])):
    face_load[0,i]=fem_info[4][i]
    #print("***face_load***:",face_load[0,i])

In [32]:
print(face_load)

In [33]:
#Face_ELEMENT
#face_load_element=np.zeros(len(fem_info[4]),dtype="int32")
face_load_element=sparse.lil_matrix((1, len(fem_info[5])),dtype="int32")
for i in range(len(fem_info[5])):
    face_load[0,i]=fem_info[5][i]
    #print("***face_load_element***:",face_load_element[0,i])

In [34]:
print(face_load)

# Dマトリクス

D行列はひずみと応力を結びつける構成式$\{\sigma\}=[D]\{\varepsilon\}$

3次元のD行列は、
$$
D=\frac{E}{(1+\nu)(1-2\nu)}\begin{Bmatrix}
   1-\nu & \nu & \nu & 0 & 0 & 0 \\
   \nu & 1-\nu & \nu & 0 & 0 & 0 \\
   \nu & \nu & 1-\nu & 0 & 0 & 0 \\
   0 & 0 & 0 & \frac{1-2\nu}{2} & 0 & 0\\ 
   0 & 0 & 0 & 0 & \frac{1-2\nu}{2} & 0\\ 
   0 & 0 & 0 & 0 & 0 & \frac{1-2\nu}{2}\\ 
\end{Bmatrix}
$$


In [35]:
#D=np.zeros((COMPONENTS, COMPONENTS),dtype="float32")
#coef = YOUNG / (1 - 2*POISSON) / (1 + POISSON)
#D=np.array([
#            [coef*(1-POISSON), coef*POISSON, coef*POISSON, 0, 0, 0],
#            [coef*POISSON, coef*(1-POISSON), coef*POISSON, 0, 0, 0],
#            [coef*POISSON, coef*POISSON, coef*(1-POISSON), 0, 0, 0],
#            [0, 0, 0, coef*(1-2*POISSON)/2, 0, 0],
#            [0, 0, 0, 0, coef*(1-2*POISSON)/2, 0],
#            [0, 0, 0, 0, 0, coef*(1-2*POISSON)/2]
#])

coef = YOUNG / (1.0 - 2.0*POISSON) / (1.0 + POISSON)
D_matrix=([
    [coef*(1.0-POISSON), coef*POISSON, coef*POISSON, 0.0, 0.0, 0.0],
    [coef*POISSON, coef*(1.0-POISSON), coef*POISSON, 0.0, 0.0, 0.0],
    [coef*POISSON, coef*POISSON, coef*(1.0-POISSON), 0.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, coef*(1.0-2.0*POISSON)/2.0, 0.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, coef*(1.0-2.0*POISSON)/2.0, 0.0],
    [0.0, 0.0, 0.0, 0.0, 0.0, coef*(1.0-2.0*POISSON)/2.0]
    ])

D = sparse.lil_matrix(D_matrix)

In [36]:
print(D)

  (0, 0)	162444.36836699763
  (0, 1)	60082.163642588166
  (0, 2)	60082.163642588166
  (1, 0)	60082.163642588166
  (1, 1)	162444.36836699763
  (1, 2)	60082.163642588166
  (2, 0)	60082.163642588166
  (2, 1)	60082.163642588166
  (2, 2)	162444.36836699763
  (3, 3)	51181.10236220473
  (4, 4)	51181.10236220473
  (5, 5)	51181.10236220473


# Bマトリクス

体要素１次要素の以下の形状関数が求まる。

- $N_{1}(x,y,z)=a_{1}+b_{1}x+c_{1}y+d_{1}z$
- $N_{2}(x,y,z)=a_{2}+b_{2}x+c_{2}y+d_{2}z$
- $N_{3}(x,y,z)=a_{3}+b_{3}x+c_{3}y+d_{3}z$
- $N_{4}(x,y,z)=a_{4}+b_{4}x+c_{4}y+d_{4}z$


形状関数$N_{1}(x,y),N_{2}(x,y),N_{3}(x,y)$を使って、

- $u=N_{1}u_{1}+N_{2}u_{2}+N_{3}u_{3}+N_{4}u_{4}$
- $v=N_{1}v_{1}+N_{2}v_{2}+N_{3}v_{3}+N_{4}v_{4}$
- $w=N_{1}w_{1}+N_{2}w_{2}+N_{3}w_{3}+N_{4}w_{4}$

ひずみから変位を求める

$$\varepsilon = \frac{\partial \boldsymbol{u}}{\partial  \boldsymbol{x}}$$

$$
\begin{Bmatrix}
   \varepsilon_{x} \\
   \varepsilon_{y} \\
   \varepsilon_{y} \\   
   \gamma_{xy} \\
   \gamma_{yz} \\
   \gamma_{zx} 
\end{Bmatrix}=
\begin{Bmatrix}
   \frac{\partial u}{\partial x} \\
   \frac{\partial v}{\partial y} \\
   \frac{\partial w}{\partial z} \\
   \frac{\partial u}{\partial y} +\frac{\partial v}{\partial x} \\
   \frac{\partial v}{\partial z} +\frac{\partial w}{\partial y} \\
   \frac{\partial w}{\partial x} +\frac{\partial u}{\partial z} 
\end{Bmatrix}
=\begin{bmatrix}
\frac{\partial N_{1}}{\partial x} & 0 &0 &  \frac{\partial N_{2}}{\partial x} & 0 &0 & \frac{\partial N_{3}}{\partial x} & 0 &0 & \frac{\partial N_{4}}{\partial x} & 0 &0\\
0 & \frac{\partial N_{1}}{\partial y} & 0 & 0 &  \frac{\partial N_{2}}{\partial y} & 0 &0 & \frac{\partial N_{3}}{\partial y} &0 & 0&\frac{\partial N_{4}}{\partial y} &0\\
0 & 0 & \frac{\partial N_{1}}{\partial z} & 0 & 0 &  \frac{\partial N_{2}}{\partial z} & 0 &0 & \frac{\partial N_{3}}{\partial z} &0 &0 &\frac{\partial N_{4}}{\partial z}\\
\frac{\partial N_{1}}{\partial y} & \frac{\partial N_{1}}{\partial x} &  0 & \frac{\partial N_{2}}{\partial y} & \frac{\partial N_{2}}{\partial x} & 0 & \frac{\partial N_{3}}{\partial y} & \frac{\partial N_{3}}{\partial x} &0 & \frac{\partial N_{4}}{\partial y} & \frac{\partial N_{4}}{\partial x} &0\\
0 &\frac{\partial N_{1}}{\partial z} & \frac{\partial N_{1}}{\partial y} &  0 & \frac{\partial N_{2}}{\partial z} & \frac{\partial N_{2}}{\partial y} & 0 & \frac{\partial N_{3}}{\partial z} & \frac{\partial N_{3}}{\partial y} &0 & \frac{\partial N_{4}}{\partial z} & \frac{\partial N_{4}}{\partial y} \\
\frac{\partial N_{1}}{\partial z} & 0 &\frac{\partial N_{1}}{\partial x} & \frac{\partial N_{2}}{\partial z} & 0 & \frac{\partial N_{2}}{\partial x} &   \frac{\partial N_{3}}{\partial z} & 0 & \frac{\partial N_{3}}{\partial x} & \frac{\partial N_{4}}{\partial z} &0 &\frac{\partial N_{4}}{\partial x}\\
\end{bmatrix}
\begin{Bmatrix}
   u_{1} \\
   v_{1} \\
   w_{1} \\
   u_{2} \\
   v_{2} \\
   w_{2} \\
   u_{3} \\
   v_{3} \\
   w_{3} \\
   u_{4} \\
   v_{4} \\
   w_{4} \\
\end{Bmatrix}
$$

$$
B=\begin{bmatrix}
\frac{\partial N_{1}}{\partial x} & 0 &0 &  \frac{\partial N_{2}}{\partial x} & 0 &0 & \frac{\partial N_{3}}{\partial x} & 0 &0 & \frac{\partial N_{4}}{\partial x} & 0 &0\\
0 & \frac{\partial N_{1}}{\partial y} & 0 & 0 &  \frac{\partial N_{2}}{\partial y} & 0 &0 & \frac{\partial N_{3}}{\partial y} &0 & 0&\frac{\partial N_{4}}{\partial y} &0\\
0 & 0 & \frac{\partial N_{1}}{\partial z} & 0 & 0 &  \frac{\partial N_{2}}{\partial z} & 0 &0 & \frac{\partial N_{3}}{\partial z} &0 &0 &\frac{\partial N_{4}}{\partial z}\\
\frac{\partial N_{1}}{\partial y} & \frac{\partial N_{1}}{\partial x} &  0 & \frac{\partial N_{2}}{\partial y} & \frac{\partial N_{2}}{\partial x} & 0 & \frac{\partial N_{3}}{\partial y} & \frac{\partial N_{3}}{\partial x} &0 & \frac{\partial N_{4}}{\partial y} & \frac{\partial N_{4}}{\partial x} &0\\
0 &\frac{\partial N_{1}}{\partial z} & \frac{\partial N_{1}}{\partial y} &  0 & \frac{\partial N_{2}}{\partial z} & \frac{\partial N_{2}}{\partial y} & 0 & \frac{\partial N_{3}}{\partial z} & \frac{\partial N_{3}}{\partial y} &0 & \frac{\partial N_{4}}{\partial z} & \frac{\partial N_{4}}{\partial y} \\
\frac{\partial N_{1}}{\partial z} & 0 &\frac{\partial N_{1}}{\partial x} & \frac{\partial N_{2}}{\partial z} & 0 & \frac{\partial N_{2}}{\partial x} &   \frac{\partial N_{3}}{\partial z} & 0 & \frac{\partial N_{3}}{\partial x} & \frac{\partial N_{4}}{\partial z} &0 &\frac{\partial N_{4}}{\partial x}\\
\end{bmatrix}
$$

しかし、以下のように全体座標系から正規化座標系に変換しておくと便利である。

<img src="img/3次元1次要素正規化.PNG">

ここで紹介している2次要素は**セレンディピティ要素**(中央に節点がない要素)。中央に節点がある要素は**ラグランジュ要素**と呼ばれる。

このとき、四角形1次要素に対して$a,b,c$を変数とした以下の形状関数を定義することができる。

2次要素で記述する。

- $N_{1}(a, b,c)=1-a-b-c$
- $N_{2}(a, b,c)=a$
- $N_{3}(a, b,c)=b$
- $N_{4}(a, b,c)=c$

例えば、$(a,,b,c)=(0,0,0)$の節点に対しては、正規化座標系では$u(0,0,0)=\sum_{i=1}^{4}N_{i}(0,0,0)u_{i}=N_{1}(0,0,0)u_{1}+N_{2}(0,0,0)u_{2}+N_{3}(0,0,0)u_{3}+N_{4}(0,0,0)u_{4}=u_{1}$となり節点番号1に対応する要素を通ることが確認できる。

Bマトリクスを求めるためには$\frac{\partial N_{i}}{\partial x},\frac{\partial N_{i}}{\partial y},\frac{\partial N_{i}}{\partial y}$を計算する必要があるが、$N_{i}(a,b,c)$は$a,b, c$を変数に持つ関数である。

$N_{i}(a, b, c)$とみなすことで、$\frac{\partial N_{i}}{\partial x},\frac{\partial N_{i}}{\partial y}, \frac{\partial N_{i}}{\partial c}$を偏微分の連鎖律より計算できる。

$$\frac{\partial N_{i}}{\partial a}=\frac{\partial N_{i}}{\partial x}\frac{\partial x}{\partial a}+\frac{\partial N_{i}}{\partial y}\frac{\partial y}{\partial a}+\frac{\partial N_{i}}{\partial z}\frac{\partial z}{\partial a}$$

$$\frac{\partial N_{i}}{\partial b}=\frac{\partial N_{i}}{\partial x}\frac{\partial x}{\partial b}+\frac{\partial N_{i}}{\partial y}\frac{\partial y}{\partial b}+\frac{\partial N_{i}}{\partial z}\frac{\partial z}{\partial b}$$

$$\frac{\partial N_{i}}{\partial c}=\frac{\partial N_{i}}{\partial x}\frac{\partial x}{\partial c}+\frac{\partial N_{i}}{\partial y}\frac{\partial y}{\partial c}+\frac{\partial N_{i}}{\partial z}\frac{\partial z}{\partial c}$$

行列でまとめると、

$$
\begin{Bmatrix}
   \frac{\partial N_{i}}{\partial a} \\
   \frac{\partial N_{i}}{\partial b} \\
   \frac{\partial N_{i}}{\partial c} 
\end{Bmatrix}
=\begin{bmatrix}
\frac{\partial x}{\partial a} & \frac{\partial y}{\partial a} & \frac{\partial z}{\partial a}\\
\frac{\partial x}{\partial b} & \frac{\partial y}{\partial b} & \frac{\partial z}{\partial b}\\
\frac{\partial x}{\partial c} & \frac{\partial y}{\partial c} & \frac{\partial z}{\partial c}\\
\end{bmatrix}
\begin{Bmatrix}
\frac{\partial N_{i}}{\partial x}\\
\frac{\partial N_{i}}{\partial y} \\
\frac{\partial N_{i}}{\partial z}
\end{Bmatrix}
$$

以下のように、ヤコビ行列を定義する。
$$J=\begin{bmatrix}
\frac{\partial x}{\partial a} & \frac{\partial y}{\partial a} & \frac{\partial z}{\partial a}\\
\frac{\partial x}{\partial b} & \frac{\partial y}{\partial b} & \frac{\partial z}{\partial b}\\
\frac{\partial x}{\partial c} & \frac{\partial y}{\partial c} & \frac{\partial z}{\partial c}\\
\end{bmatrix}$$

右から$J$の逆行列を作用させると、

$$
\begin{Bmatrix}
\frac{\partial N_{i}}{\partial x}\\
\frac{\partial N_{i}}{\partial y} \\
\frac{\partial N_{i}}{\partial z}
\end{Bmatrix}
= 
J^{-1}
\begin{Bmatrix}
   \frac{\partial N_{i}}{\partial a} \\
   \frac{\partial N_{i}}{\partial b} \\
   \frac{\partial N_{i}}{\partial c} 
\end{Bmatrix}
$$

以上のようにすると、$\frac{\partial N_{i}}{\partial x},\frac{\partial N_{i}}{\partial y}, \frac{\partial N_{i}}{\partial z}$を計算できる。

四面体中の任意の$(x,y,z)$の座標値について、

- $x=N_{1}x_{1}+N_{2}x_{2}+N_{3}x_{3}+N_{4}x_{4}$
- $y=N_{1}y_{1}+N_{2}y_{2}+N_{3}y_{3}+N_{4}y_{4}$
- $z=N_{1}z_{1}+N_{2}z_{2}+N_{3}z_{3}+N_{4}z_{4}$

- $\frac{\partial x}{\partial a}=\sum_{i=1}^{4}\frac{\partial N_{i}}{\partial a}x_{i}$
- $\frac{\partial y}{\partial a}=\sum_{i=1}^{4}\frac{\partial N_{i}}{\partial a}y_{i}$
- $\frac{\partial z}{\partial a}=\sum_{i=1}^{4}\frac{\partial N_{i}}{\partial a}x_{i}$

- $\frac{\partial x}{\partial b}=\sum_{i=1}^{4}\frac{\partial N_{i}}{\partial b}x_{i}$
- $\frac{\partial y}{\partial b}=\sum_{i=1}^{4}\frac{\partial N_{i}}{\partial b}y_{i}$
- $\frac{\partial z}{\partial b}=\sum_{i=1}^{4}\frac{\partial N_{i}}{\partial b}x_{i}$

- $\frac{\partial x}{\partial c}=\sum_{i=1}^{4}\frac{\partial N_{i}}{\partial c}x_{i}$
- $\frac{\partial y}{\partial c}=\sum_{i=1}^{4}\frac{\partial N_{i}}{\partial c}y_{i}$
- $\frac{\partial z}{\partial c}=\sum_{i=1}^{4}\frac{\partial N_{i}}{\partial c}x_{i}$

- $\frac{\partial N_{1}}{\partial a}=-1$
- $\frac{\partial N_{2}}{\partial a}=1$
- $\frac{\partial N_{3}}{\partial a}=0$
- $\frac{\partial N_{4}}{\partial a}=0$
- $\frac{\partial N_{1}}{\partial b}=-1$
- $\frac{\partial N_{2}}{\partial b}=0$
- $\frac{\partial N_{3}}{\partial b}=1$
- $\frac{\partial N_{4}}{\partial b}=0$
- $\frac{\partial N_{1}}{\partial c}=-1$
- $\frac{\partial N_{2}}{\partial c}=0$
- $\frac{\partial N_{3}}{\partial c}=0$
- $\frac{\partial N_{4}}{\partial c}=1$

## プログラムの流れを整理

ヤコビ行列を計算

$$J=\begin{bmatrix}
\frac{\partial x}{\partial a} & \frac{\partial y}{\partial a} & \frac{\partial z}{\partial a}\\
\frac{\partial x}{\partial b} & \frac{\partial y}{\partial b} & \frac{\partial z}{\partial b}\\
\frac{\partial x}{\partial c} & \frac{\partial y}{\partial c} & \frac{\partial z}{\partial c}\\
\end{bmatrix}$$

次に$i=1$~$4$に対して以下を計算

$$
\begin{Bmatrix}
\frac{\partial N_{i}}{\partial x}\\
\frac{\partial N_{i}}{\partial y} \\
\frac{\partial N_{i}}{\partial z}
\end{Bmatrix}
= 
J^{-1}
\begin{Bmatrix}
   \frac{\partial N_{i}}{\partial a} \\
   \frac{\partial N_{i}}{\partial b} \\
   \frac{\partial N_{i}}{\partial c} 
\end{Bmatrix}
$$

これによりBマトリクスが計算できる。

$$
B=\begin{bmatrix}
\frac{\partial N_{1}}{\partial x} & 0 &0 &  \frac{\partial N_{2}}{\partial x} & 0 &0 & \frac{\partial N_{3}}{\partial x} & 0 &0 & \frac{\partial N_{4}}{\partial x} & 0 &0\\
0 & \frac{\partial N_{1}}{\partial y} & 0 & 0 &  \frac{\partial N_{2}}{\partial y} & 0 &0 & \frac{\partial N_{3}}{\partial y} &0 & 0&\frac{\partial N_{4}}{\partial y} &0\\
0 & 0 & \frac{\partial N_{1}}{\partial z} & 0 & 0 &  \frac{\partial N_{2}}{\partial z} & 0 &0 & \frac{\partial N_{3}}{\partial z} &0 &0 &\frac{\partial N_{4}}{\partial z}\\
\frac{\partial N_{1}}{\partial y} & \frac{\partial N_{1}}{\partial x} &  0 & \frac{\partial N_{2}}{\partial y} & \frac{\partial N_{2}}{\partial x} & 0 & \frac{\partial N_{3}}{\partial y} & \frac{\partial N_{3}}{\partial x} &0 & \frac{\partial N_{4}}{\partial y} & \frac{\partial N_{4}}{\partial x} &0\\
0 &\frac{\partial N_{1}}{\partial z} & \frac{\partial N_{1}}{\partial y} &  0 & \frac{\partial N_{2}}{\partial z} & \frac{\partial N_{2}}{\partial y} & 0 & \frac{\partial N_{3}}{\partial z} & \frac{\partial N_{3}}{\partial y} &0 & \frac{\partial N_{4}}{\partial z} & \frac{\partial N_{4}}{\partial y} \\
\frac{\partial N_{1}}{\partial z} & 0 &\frac{\partial N_{1}}{\partial x} & \frac{\partial N_{2}}{\partial z} & 0 & \frac{\partial N_{2}}{\partial x} &   \frac{\partial N_{3}}{\partial z} & 0 & \frac{\partial N_{3}}{\partial x} & \frac{\partial N_{4}}{\partial z} &0 &\frac{\partial N_{4}}{\partial x}\\
\end{bmatrix}
$$

In [70]:
B = np.zeros((ELEMENTS, COMPONENTS, DOF_TRIA3),dtype="float32")
Jmat = np.zeros((ELEMENTS, DOF_NODE, DOF_NODE),dtype="float32")  

In [71]:

for e in range(ELEMENTS):
    x0, y0, z0 = x[0, connectivity[e,0]], y[0, connectivity[e,0]], z[0, connectivity[e,0]]
    x1, y1, z1 = x[0, connectivity[e,1]], y[0, connectivity[e,1]], z[0, connectivity[e,1]]
    x2, y2, z2 = x[0, connectivity[e,2]], y[0, connectivity[e,2]], z[0, connectivity[e,2]]
    x3, y3, z3 = x[0, connectivity[e,3]], y[0, connectivity[e,3]], z[0, connectivity[e,3]]
    #print(f"x0:{x0}, y0:{y0}, z0:{z0}")

    #➀形状関数の正規化座標による偏微分
    # N1=1-a-b-c, N2=a, N3=b, N4=c 
    dN1da = -1.0; dN2da = 1.0; dN3da = 0.0; dN4da = 0.0
    dN1db = -1.0; dN2db = 0.0; dN3db = 1.0; dN4db = 0.0
    dN1dc = -1.0; dN2dc = 0.0; dN3dc = 0.0; dN4dc = 1.0

    #➁座標成分を正規化座標成分で偏微分
    dxda = dN1da*x0 + dN2da*x1 + dN3da*x2 + dN4da*x3
    dyda = dN1da*y0 + dN2da*y1 + dN3da*y2 + dN4da*y3
    dzda = dN1da*z0 + dN2da*z1 + dN3da*z2 + dN4da*z3
    dxdb = dN1db*x0 + dN2db*x1 + dN3db*x2 + dN4db*x3
    dydb = dN1db*y0 + dN2db*y1 + dN3db*y2 + dN4db*y3
    dzdb = dN1db*z0 + dN2db*z1 + dN3db*z2 + dN4db*z3
    dxdc = dN1dc*x0 + dN2dc*x1 + dN3dc*x2 + dN4dc*x3
    dydc = dN1dc*y0 + dN2dc*y1 + dN3dc*y2 + dN4dc*y3
    dzdc = dN1dc*z0 + dN2dc*z1 + dN3dc*z2 + dN4dc*z3

    # ➂ヤコビ行列Jmat
    Jmat[e]=np.array([
                    [dxda, dyda, dzda],
                    [dxdb, dydb, dzdb],
                    [dxdc, dydc, dzdc]
                    ])
    
    # ∂Ｎｉ／∂ａ、∂Ｎｉ／∂ｂ、∂Ｎｉ／∂ｃ
    dN1dabc = [dN1da, dN1db, dN1dc]; dN2dabc = [dN2da, dN2db, dN2dc]; dN3dabc = [dN3da, dN3db, dN3dc]; dN4dabc = [dN4da, dN4db, dN4dc]
    
    # ∂Ｎｉ／∂ｘ、∂Ｎｉ／∂ｙ、∂Ｎｉ／∂ｚ
    # dN1dxyz = Jmat[e].inv @ dN1dabc
    dN1dxyz = LA.solve(Jmat[e], dN1dabc); dN2dxyz = LA.solve(Jmat[e], dN2dabc); dN3dxyz = LA.solve(Jmat[e], dN3dabc); dN4dxyz = LA.solve(Jmat[e], dN4dabc)

    # 要素e番目の行列
    dN1dx = dN1dxyz[0]; dN2dx = dN2dxyz[0]; dN3dx = dN3dxyz[0]; dN4dx = dN4dxyz[0]
    dN1dy = dN1dxyz[1]; dN2dy = dN2dxyz[1]; dN3dy = dN3dxyz[1]; dN4dy = dN4dxyz[1]
    dN1dz = dN1dxyz[2]; dN2dz = dN2dxyz[2]; dN3dz = dN3dxyz[2]; dN4dz = dN4dxyz[2]
    
    B[e] =  np.array([
                    [dN1dx, 0.0, 0.0, dN2dx, 0.0, 0.0, dN3dx, 0.0, 0.0, dN4dx, 0.0, 0.0],
                    [0.0, dN1dy, 0.0, 0.0, dN2dy, 0.0, 0.0, dN3dy, 0.0, 0.0, dN4dy, 0.0],
                    [0.0, 0.0, dN1dz, 0.0, 0.0, dN2dz, 0.0, 0.0, dN3dz, 0.0, 0.0, dN4dz],
                    [0.0, dN1dz, dN1dy, 0.0, dN2dz, dN2dy, 0.0, dN3dz, dN3dy, 0.0, dN4dz, dN4dy],
                    [dN1dz, 0.0, dN1dx, dN2dz, 0.0, dN2dx, dN3dz, 0.0, dN3dx, dN4dz, 0.0, dN4dx],
                    [dN1dy, dN1dx, 0.0, dN2dy, dN2dx, 0.0, dN3dy, dN3dx, 0.0, dN4dy, dN4dx, 0.0]
                    ])
    # print("Pass:", e)
    
"""
print("==============  Bマトリクス ========================")
for e in range(ELEMENTS):
    print(f"======= 要素{e}番目==========")
    print(f"         B[{e}] = {B[e]}")
"""

'\nprint("==============  Bマトリクス ========================")\nfor e in range(ELEMENTS):\n    print(f"======= 要素{e}番目==========")\n    print(f"         B[{e}] = {B[e]}")\n'

In [72]:
dN1dx,dN1dy, dN1dz

(0.00020297389235035052, 0.12773239926663219, 0.00796822221023457)

In [73]:
dN2dx,dN2dy, dN2dz

(0.003937835775053984, -0.11000580103054382, -0.11370302428663463)

In [74]:
dN3dx,dN3dy, dN3dz

(0.10288128431346771, 0.08939471148474638, 0.1178657503612472)

In [75]:
dN4dx,dN4dy, dN4dz

(-0.10702209398087201, -0.10712130972083472, -0.012130948284847138)

In [76]:
np.set_printoptions(linewidth=200)

In [77]:
print("==============  Bマトリクス ========================")
print(f"======= 要素{ELEMENTS-1}番目==========")
print(f"         B[{ELEMENTS-1}] = \n{B[ELEMENTS-1]}")

==============  Bマトリクス ========================
======= 要素1402番目==========
         B[1402] = 
[[ 0.00020297  0.          0.          0.00393784  0.          0.          0.10288128  0.          0.         -0.10702209  0.          0.        ]
 [ 0.          0.1277324   0.          0.         -0.1100058   0.          0.          0.08939471  0.          0.         -0.10712131  0.        ]
 [ 0.          0.          0.00796822  0.          0.         -0.11370303  0.          0.          0.11786575  0.          0.         -0.01213095]
 [ 0.          0.00796822  0.1277324   0.         -0.11370303 -0.1100058   0.          0.11786575  0.08939471  0.         -0.01213095 -0.10712131]
 [ 0.00796822  0.          0.00020297 -0.11370303  0.          0.00393784  0.11786575  0.          0.10288128 -0.01213095  0.         -0.10702209]
 [ 0.1277324   0.00020297  0.         -0.1100058   0.00393784  0.          0.08939471  0.10288128  0.         -0.10712131 -0.10702209  0.        ]]


Bマトリクスは要素ごとに値が格納されている

In [78]:
# e=0の要素に対してB[0]
print(B[0])

[[-0.2         0.          0.          0.2         0.          0.          0.          0.          0.          0.          0.          0.        ]
 [ 0.         -0.15117748  0.          0.          0.          0.          0.          0.15117748  0.          0.          0.          0.        ]
 [ 0.          0.         -0.11594871  0.          0.          0.          0.          0.          0.          0.          0.          0.11594871]
 [ 0.         -0.11594871 -0.15117748  0.          0.          0.          0.          0.          0.15117748  0.          0.11594871  0.        ]
 [-0.11594871  0.         -0.2         0.          0.          0.2         0.          0.          0.          0.11594871  0.          0.        ]
 [-0.15117748 -0.2         0.          0.          0.2         0.          0.15117748  0.          0.          0.          0.          0.        ]]


# 要素後剛性マトリクス

In [79]:
weight = 1.0 / 6.0                                          #積分点の重み係数

要素剛性マトリクス
$$[K_{e}]=\int\int\int\, [B]^{T}[D]^{T}[B]\,dx\,dy\,dz$$

ヤコビ行列式$|J|$を用いて$a,b,c$の変数に変換。

$$[k_{e}]=\int\int\int\, [B]^{T}[D]^{T}[B]\,|J|\,da\,db\,dc$$


$$[K_{e}]=\int^{1}_{0}\int^{1}_{0}\int^{1}_{0}\, [B]^{T}[D]^{T}[B]\,|J|\,da\,db\,dc = w\big([B(a,b,c)]^{T}[D]^{T}[B(a,b,c)]\big))$$

$w=\frac{1}{6}$

In [80]:
Ke= np.zeros((ELEMENTS,  DOF_TRIA3, DOF_TRIA3),dtype="float32") 
for e in range(ELEMENTS):
    Ke[e] = weight*B[e].T @ D @ B[e] * LA.det(Jmat[e])
    #print(Ke[e])

In [81]:
np.set_printoptions(threshold=np.inf,suppress=True, precision=4, floatmode='maxprec')
Ke[0][0]

array([ 397230.47 ,  159931.73 ,  122662.96 , -308908.88 ,  -73568.59 ,  -56424.96 ,  -55609.574,  -86363.13 ,       0.   ,  -32712.006,       0.   ,  -66238.   ], dtype=float32)

In [82]:
print(Ke)

[[[ 397230.47    159931.73    122662.96   -308908.88    -73568.59    -56424.96    -55609.574   -86363.13         0.      -32712.006        0.      -66238.    ]
  [ 159931.73    306539.4      92719.38    -86363.13    -97327.45         0.      -73568.59   -176499.95    -42650.918        0.      -32712.006   -50068.465 ]
  [ 122662.96     92719.38    256762.1     -66238.           0.      -97327.45         0.      -50068.47    -55609.574   -56424.96    -42650.918  -103825.06  ]
  [-308908.88    -86363.13    -66238.      308908.88         0.           0.           0.       86363.13         0.           0.           0.       66238.    ]
  [ -73568.59    -97327.45         0.           0.       97327.45         0.       73568.59         0.           0.           0.           0.           0.    ]
  [ -56424.96         0.      -97327.45         0.           0.       97327.45         0.           0.           0.       56424.96         0.           0.    ]
  [ -55609.574   -73568.59         0.   

# 全体剛性マトリクス

In [83]:
K= sparse.lil_matrix((DOF_TOTAL, DOF_TOTAL),dtype="float32")

for e in range(ELEMENTS):
    for r in range(DOF_TRIA3):
        rt = ((connectivity[e, r // DOF_NODE]+1)*DOF_NODE-((r+1)%DOF_NODE))-1
        #print("rt:",rt)
        for c in range(DOF_TRIA3):
            ct = ((connectivity[e, c // DOF_NODE]+1)*DOF_NODE-((c+1)%DOF_NODE))-1
            K[rt,ct] = K[rt,ct] + Ke[e,r,c]

In [84]:
1//3

0

In [85]:
K= sparse.lil_matrix((DOF_TOTAL, DOF_TOTAL),dtype="float32")

for e in range(0,3):
    for r in range(DOF_TRIA3):
        rt = ((connectivity[e, r // DOF_NODE]+1)*DOF_NODE-((r+1)%DOF_NODE))-1
        print("rt:",rt)
        for c in range(DOF_TRIA3):
            ct = ((connectivity[e, c // DOF_NODE]+1)*DOF_NODE-((c+1)%DOF_NODE))-1
            K[rt,ct] = K[rt,ct] + Ke[e,r,c]

rt: 1
rt: 0
rt: 2
rt: 70
rt: 69
rt: 71
rt: 265
rt: 264
rt: 266
rt: 25
rt: 24
rt: 26
rt: 277
rt: 276
rt: 278
rt: 70
rt: 69
rt: 71
rt: 25
rt: 24
rt: 26
rt: 871
rt: 870
rt: 872
rt: 841
rt: 840
rt: 842
rt: 265
rt: 264
rt: 266
rt: 70
rt: 69
rt: 71
rt: 871
rt: 870
rt: 872


In [86]:
connectivity[0,1]

23

In [87]:
print(K.toarray()[:10])

[[ 306539.4    159931.73    92719.38        0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.
        0.          0.          0.          0.          0.          0.          0.          0.     -32712.006       0.     -50068.465       0.          0.          0.          0.          0.
        0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.
        0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.          0.
        0.          0.          0.          0.          0.     -97327.45   -86363.13        0.          0.          0.          0.          0.          0.          0.          0.          0.
        0.          0.          0.          0

行列の要素の値0のものは格納されていない。

# 荷重

In [88]:
#F=np.zeros(DOF_TOTAL,dtype="float32")
F=sparse.lil_matrix((1, DOF_TOTAL),dtype="float32")

#for i in range(DOF_TOTAL):
#    F[0, i] = 0.0

#note: forceは１点としている
F[0, force[0,0]*DOF_NODE] += Fx    
F[0, force[0,0]*DOF_NODE+1] += Fy
F[0, force[0,0]*DOF_NODE+2] += Fz

In [89]:
force[0,0], DOF_NODE, DOF_TOTAL

(282, 3, 1275)

In [90]:
Fx, Fy, Fz

(-100.0, 0.0, 0.0)

In [91]:
print(F)

  (0, 846)	-100.0


# 境界条件

In [92]:
print(U)

In [93]:
# 全体行列をコピー
#Kc = sparse.lil_matrix.copy(K)

for r in range(DOF_TOTAL):
    if Um[r] == True:
        for rr in range(DOF_TOTAL): # 変位拘束が存在する行の成分を０にする
            if rr != r:
                F[0, rr] -= K[rr,r]*U[0, r]   
                if F[0,rr] !=0 : print(rr, F[0, rr])
        for rr in range(DOF_TOTAL):
            K[rr,r] = 0.0
        for cc in range(DOF_TOTAL): # 変位拘束が存在する列の成分を０にする
            K[r,cc] = 0.0

        K[r,r] = 1.0 #対角成分を１

        #F[r] = U[r]
        F[0, r] = U[0, r]

846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0
846 -100.0


In [94]:
K.shape

(1275, 1275)

In [95]:
print(K)

  (0, 0)	1.0
  (1, 1)	1.0
  (2, 2)	1.0
  (6, 6)	1.0
  (7, 7)	1.0
  (8, 8)	1.0
  (12, 12)	1.0
  (13, 13)	1.0
  (14, 14)	1.0
  (18, 18)	1.0
  (19, 19)	1.0
  (20, 20)	1.0
  (24, 24)	71241.43
  (24, 25)	-6354.752
  (24, 26)	1656.2087
  (24, 276)	-58387.18
  (24, 277)	56233.418
  (24, 278)	-10334.477
  (24, 870)	12881.476
  (24, 871)	-49094.54
  (24, 872)	12795.275
  (25, 24)	-6354.752
  (25, 25)	147586.86
  (25, 26)	-20034.46
  (25, 276)	64357.03
  (25, 277)	-164396.95
  (25, 278)	-22251.996
  (25, 870)	-57632.723
  (25, 871)	4058.547
  (26, 24)	1656.2087
  (26, 25)	-20034.46
  (26, 26)	147050.64
  (26, 276)	-13091.897
  (26, 277)	-14507.728
  (26, 278)	-31800.475
  (26, 870)	15020.541
  (26, 872)	4058.547
  (69, 69)	1.0
  (70, 70)	1.0
  (71, 71)	1.0
  (72, 72)	1.0
  (73, 73)	1.0
  (74, 74)	1.0
  (75, 75)	1.0
  (76, 76)	1.0
  (77, 77)	1.0
  (132, 132)	1.0
  (133, 133)	1.0
  (134, 134)	1.0
  (135, 135)	1.0
  (136, 136)	1.0
  (137, 137)	1.0
  (138, 138)	1.0
  (139, 139)	1.0
  (140, 140)	1.0


In [96]:
print(K.toarray()[:10])

[[1. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.
  0. 0. 0. 0. 0

In [97]:
Um

array([ True,  True,  True, False, False, False,  True,  True,  True, False, False, False,  True,  True,  True, False, False, False,  True,  True,  True, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,  True,  True,  True,  True,  True,  True,  True,  True,  True, False, False, False,
       False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,
       False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False, False,  True,  True,  True,
        True,  True,

In [101]:
print(K)

  (0, 0)	1.0
  (1, 1)	1.0
  (2, 2)	1.0
  (6, 6)	1.0
  (7, 7)	1.0
  (8, 8)	1.0
  (12, 12)	1.0
  (13, 13)	1.0
  (14, 14)	1.0
  (18, 18)	1.0
  (19, 19)	1.0
  (20, 20)	1.0
  (24, 24)	71241.43
  (24, 25)	-6354.752
  (24, 26)	1656.2087
  (24, 276)	-58387.18
  (24, 277)	56233.418
  (24, 278)	-10334.477
  (24, 870)	12881.476
  (24, 871)	-49094.54
  (24, 872)	12795.275
  (25, 24)	-6354.752
  (25, 25)	147586.86
  (25, 26)	-20034.46
  (25, 276)	64357.03
  (25, 277)	-164396.95
  (25, 278)	-22251.996
  (25, 870)	-57632.723
  (25, 871)	4058.547
  (26, 24)	1656.2087
  (26, 25)	-20034.46
  (26, 26)	147050.64
  (26, 276)	-13091.897
  (26, 277)	-14507.728
  (26, 278)	-31800.475
  (26, 870)	15020.541
  (26, 872)	4058.547
  (69, 69)	1.0
  (70, 70)	1.0
  (71, 71)	1.0
  (72, 72)	1.0
  (73, 73)	1.0
  (74, 74)	1.0
  (75, 75)	1.0
  (76, 76)	1.0
  (77, 77)	1.0
  (132, 132)	1.0
  (133, 133)	1.0
  (134, 134)	1.0
  (135, 135)	1.0
  (136, 136)	1.0
  (137, 137)	1.0
  (138, 138)	1.0
  (139, 139)	1.0
  (140, 140)	1.0


# 逆行列を計算

In [103]:
K_csc = K.tocsc()
F_csc = F.tocsc()

In [105]:
print(K_csc)

  (0, 0)	1.0
  (1, 1)	1.0
  (2, 2)	1.0
  (6, 6)	1.0
  (7, 7)	1.0
  (8, 8)	1.0
  (12, 12)	1.0
  (13, 13)	1.0
  (14, 14)	1.0
  (18, 18)	1.0
  (19, 19)	1.0
  (20, 20)	1.0
  (24, 24)	71241.43
  (25, 24)	-6354.752
  (26, 24)	1656.2087
  (276, 24)	-58387.18
  (277, 24)	56233.414
  (278, 24)	-10334.477
  (870, 24)	12881.476
  (871, 24)	-49094.54
  (872, 24)	12795.276
  (24, 25)	-6354.752
  (25, 25)	147586.86
  (26, 25)	-20034.46
  (276, 25)	64357.027
  :	:
  (835, 835)	1.0
  (836, 836)	1.0
  (837, 837)	1.0
  (838, 838)	1.0
  (839, 839)	1.0
  (840, 840)	1.0
  (841, 841)	1.0
  (842, 842)	1.0
  (24, 870)	12881.476
  (25, 870)	-57632.723
  (26, 870)	15020.541
  (276, 870)	-225141.28
  (277, 870)	86363.14
  (278, 870)	50068.47
  (870, 870)	272531.9
  (24, 871)	-49094.54
  (25, 871)	4058.547
  (276, 871)	73568.6
  (277, 871)	-70934.92
  (871, 871)	124351.95
  (24, 872)	12795.275
  (26, 872)	4058.547
  (276, 872)	42650.918
  (278, 872)	-70934.92
  (872, 872)	246502.31


In [106]:
Inv_K_csc = inv(K_csc)

RuntimeError: Factor is exactly singular

In [107]:
print(Inv_K_csc)

  (0, 0)	1.0
  (1, 1)	1.0
  (2, 2)	1.0
  (3, 3)	0.000119343276
  (4, 3)	-4.596578e-06
  (5, 3)	1.7497197e-05
  (9, 3)	0.00010538387
  (10, 3)	-2.7629699e-06
  (11, 3)	1.5635269e-05
  (15, 3)	0.00010541942
  (16, 3)	4.7953363e-06
  (17, 3)	-1.5963904e-05
  (21, 3)	0.00011307796
  (22, 3)	4.7878502e-06
  (23, 3)	-1.569614e-05
  (24, 3)	1.063518e-06
  (25, 3)	3.6816013e-07
  (26, 3)	1.7818516e-06
  (27, 3)	2.5220988e-06
  (28, 3)	2.0358227e-07
  (29, 3)	3.1370594e-06
  (30, 3)	6.1954593e-06
  (31, 3)	-5.1655203e-08
  (32, 3)	5.295413e-06
  (33, 3)	1.0450567e-05
  :	:
  (1250, 1274)	2.5021777e-07
  (1251, 1274)	6.9957326e-07
  (1252, 1274)	4.718804e-08
  (1253, 1274)	2.0817508e-07
  (1254, 1274)	5.06229e-07
  (1255, 1274)	3.151996e-08
  (1256, 1274)	4.3148688e-07
  (1257, 1274)	4.5473763e-07
  (1258, 1274)	4.7498126e-08
  (1259, 1274)	4.9661156e-07
  (1260, 1274)	4.6200677e-07
  (1261, 1274)	2.1609674e-08
  (1262, 1274)	2.406402e-07
  (1263, 1274)	4.8596655e-07
  (1264, 1274)	3.4786304e-08

In [ ]:
FT_csc = F_csc.T

In [ ]:
del K_csc
del F_csc
gc.collect

In [ ]:
print(Inv_K_csc[:5])

# 連立方程式を解く

変位拘束と荷重拘束を加えて修正した剛性マトリクスより変位を求める。

$[K]\{U\}-\{F\} = 0$から$\{U\}$を求める。

簡単に計算するためには左から逆行列を掛けて$\{U\}=[K]^{-1}\{F\}$により計算する。

In [108]:
Ua=Inv_K_csc@FT_csc

In [109]:
print(Ua)

  (1274, 0)	-0.00030535297
  (1273, 0)	-5.9545096e-06
  (1272, 0)	-0.00046959353
  (1271, 0)	-5.6582292e-05
  (1270, 0)	-7.0176034e-06
  (1269, 0)	-0.00049437024
  (1268, 0)	-0.00031096645
  (1267, 0)	-2.5917172e-05
  (1266, 0)	-0.0005297474
  (1265, 0)	4.9122882e-05
  (1264, 0)	-1.4130366e-05
  (1263, 0)	-0.0008857001
  (1262, 0)	-8.343649e-05
  (1261, 0)	-1.304338e-05
  (1260, 0)	-0.00082377857
  (1259, 0)	-0.0003940767
  (1258, 0)	-3.8139053e-06
  (1257, 0)	-0.00086554466
  (1256, 0)	-0.00044033432
  (1255, 0)	-2.3148421e-05
  (1254, 0)	-0.0009635878
  (1253, 0)	-9.041407e-06
  (1252, 0)	-1.20236455e-05
  (1251, 0)	-0.001407002
  (1250, 0)	-0.00010760654
  :	:
  (36, 0)	-0.0013889273
  (35, 0)	-0.0006852867
  (34, 0)	-6.284885e-05
  (33, 0)	-0.0009465576
  (32, 0)	-0.000529241
  (31, 0)	-6.833762e-05
  (30, 0)	-0.00054520986
  (29, 0)	-0.0003137627
  (28, 0)	-6.325449e-05
  (27, 0)	-0.00020978437
  (26, 0)	-0.00017835999
  (25, 0)	-6.0548176e-05
  (24, 0)	-8.149162e-05
  (23, 0)	0.0

# ParaView表示のためのvtk変換

In [ ]:
import os

PWD = pathlib.Path(os.getcwd())

file_name = f"{inpfileName[-4:]}_TRIA_3.vtk"
# f_path = pathlib.Path(__file__).parent.resolve() / file_name 
f_path = PWD / file_name 

with open(f_path, mode = "w") as f:
    
    #Header出力
    print("# vtk DataFile Version 2.0",file=f)
    print("Header",file=f)
    print("ASCII",file=f)
    print("DATASET UNSTRUCTURED_GRID",file=f)
    print(" ",file=f)
    
    #節点座標出力
    print("POINTS", NODES, " double",file=f)
    for i in range(NODES):
        print(x[0,i]," ",y[0,i]," ",z[0,i],file=f)
    print(" ",file=f)
    
    #要素構成節点番号出力
    print("CELLS", ELEMENTS, ELEMENTS*5,file=f)
    for i in range(ELEMENTS):
        print(4," ",end="",file=f)
        for j in range(NODE_TRIA3):
            print(connectivity[i,j]," ",end="",file=f)
        print("",file=f)
    print(" ",file=f)
    
    #要素タイプ出力
    print("CELL_TYPES", ELEMENTS,file=f)
    for i in range(ELEMENTS):
        print(10,file=f)
    print(" ",file=f)
    
    #節点応力出力
    print("POINT_DATA", NODES,file=f)
    #print("SCALARS Sx float 1",file=f)
    #print("LOOKUP_TABLE default",file=f)
    #for i in range(NODES):
    #    print(stress_node[i,0],file=f)
    print(" ",file=f)
    
    #節点変位出力
    print("VECTORS Displacement float",file=f)
    for i in range(NODES):
        print(Ua[i*DOF_NODE,0]," ",Ua[i*DOF_NODE+1,0]," ",Ua[i*DOF_NODE+2,0],file=f)
    print(" ",file=f)

Quad4_FEM_00_TRIA_3.vtkが出力されているのでParaViewで可視化

<img src="img/3次元1次要素梁の問題Paraview.PNG">

# CATIAで作成した複雑な形状

<img src="img/3次元1次要素BEDParaview.PNG">

# Pythonで行列計算

In [29]:
import numpy as np
from scipy.linalg import solve
import time

def main():
    # 行列AとベクトルBのサイズ
    n = 10000

    # 行列AとベクトルBをランダムな値で生成
    A = np.random.rand(n, n)
    B = np.random.rand(n)

    # 解く前の時刻を記録
    start_time = time.time()

    # 線形方程式AX = Bを解く
    X = solve(A, B)

    # 解いた後の時刻を記録
    end_time = time.time()

    # 処理にかかった時間を表示
    print(f"処理時間: {end_time - start_time}秒")

    # 解の一部を表示（例えば最初の10要素）
    print("解Xの最初の10要素:", X[:10])

if __name__ == "__main__":
    main()

処理時間: 13.04245662689209秒
解Xの最初の10要素: [ 1.73589125 -0.40829237  0.33569364  1.70567012 -0.74251665  0.20523761
 -0.99312904  1.13543919  1.08412308 -1.71890438]
